# 07 — Multimodal ML Model

Phase 7 of the Indian Equity AI project plan.

Combines price/technical (Phase 3), fundamental (Phase 4), news/sentiment (Phase 6),
macro, and market-regime features into one feature vector, trains a stacking ensemble
(per-modality base models + a **learned** meta-model, not fixed weights), and validates
hypotheses H1-H5 with walk-forward folds.

**Data-source caveats carried forward:** news coverage is sparse (Phase 6: ~12 articles
total, recent window only) and fundamentals only change once per fiscal year (Phase 4:
up to 5 fiscal years per stock) -- so the news and fundamental modalities are expected to
carry a weak, low-granularity signal for a *daily next-day* target. That is reported
honestly below rather than hidden.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier

from src.ingestion.market_data import download_index_ohlcv
from src.macro.ingestion import download_macro_data
from src.macro.features import add_macro_returns
from src.regime.classifier import classify_market_regime
from src.models.dataset import chronological_split
from src.models.multimodal_dataset import MULTIMODAL_FEATURE_GROUPS, build_multimodal_dataset
from src.models.ensemble import build_meta_features, ensemble_predict_proba, fit_meta_model, fit_modality_models
from src.models.evaluate import classification_metrics
from src.models.walkforward import expanding_window_folds

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RANDOM_STATE = 42


def make_lgbm():
    return LGBMClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1,
    )

## Step 1: Download macro data and NIFTY 50, build the market-regime feature

In [2]:
macro = download_macro_data(period="5y")
macro = add_macro_returns(macro)
print("Macro rows:", len(macro), "| columns:", macro.columns.tolist())

nifty50 = download_index_ohlcv("^NSEI", period="5y")[["date", "close"]].rename(columns={"close": "nifty_close"})
regime_input = nifty50.merge(macro[["date", "india_vix"]], on="date", how="inner").sort_values("date")

regime = classify_market_regime(regime_input["nifty_close"], regime_input["india_vix"])
regime["date"] = regime_input["date"].values
regime["regime_label"] = np.select(
    [
        (regime["is_bull"] == 1) & (regime["is_high_vol"] == 0),
        (regime["is_bull"] == 1) & (regime["is_high_vol"] == 1),
        (regime["is_bull"] == 0) & (regime["is_high_vol"] == 0),
        (regime["is_bull"] == 0) & (regime["is_high_vol"] == 1),
    ],
    ["bull_low_vol", "bull_high_vol", "bear_low_vol", "bear_high_vol"],
    default="unknown",
)
regime["regime_label"].value_counts()

Macro rows: 1301 | columns: ['date', 'india_vix', 'usdinr', 'brent_crude', 'us10y_yield', 'sp500', 'india_vix_return_1d', 'usdinr_return_1d', 'brent_crude_return_1d', 'us10y_yield_return_1d', 'sp500_return_1d']


regime_label
bull_low_vol     569
bear_low_vol     249
bear_high_vol    212
bull_high_vol    142
unknown           61
Name: count, dtype: int64

## Step 2: Load Phase 3/4/6 outputs and build the multimodal dataset

In [3]:
technical = pd.read_parquet(DATA_PROCESSED / "technical_features.parquet")
fundamentals_annual = pd.read_parquet(DATA_PROCESSED / "fundamentals_annual.parquet")
news_daily = pd.read_parquet(DATA_PROCESSED / "news_daily_features.parquet")

dataset = build_multimodal_dataset(technical, fundamentals_annual, news_daily, macro, regime)

print("Rows:", len(dataset), "| Symbols:", dataset["symbol"].nunique())
print("Date range:", dataset["date"].min().date(), "to", dataset["date"].max().date())
print("Base rate:", dataset["next_day_direction"].mean().round(4))
print("Feature groups:", {k: len(v) for k, v in MULTIMODAL_FEATURE_GROUPS.items()})

Rows: 2530 | Symbols: 5
Date range: 2024-05-30 to 2026-09-24
Base rate: 0.4727
Feature groups: {'technical': 24, 'fundamental': 5, 'news': 5, 'macro_regime': 8}


## Step 3: Chronological split

In [4]:
train, val, test = chronological_split(dataset, train_frac=0.7, val_frac=0.15)
for name, split in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s}: {len(split):5d} rows | {split['date'].min().date()} to {split['date'].max().date()}")

y_train, y_val, y_test = train["next_day_direction"], val["next_day_direction"], test["next_day_direction"]

X_train_by_modality = {m: train[cols] for m, cols in MULTIMODAL_FEATURE_GROUPS.items()}
X_val_by_modality = {m: val[cols] for m, cols in MULTIMODAL_FEATURE_GROUPS.items()}
X_test_by_modality = {m: test[cols] for m, cols in MULTIMODAL_FEATURE_GROUPS.items()}

train:  1770 rows | 2024-05-30 to 2026-01-07
val  :   380 rows | 2026-01-08 to 2026-05-21
test :   380 rows | 2026-05-22 to 2026-09-24


## Step 4: Fit one base model per modality, then a learned meta-model on validation

In [5]:
modality_models = fit_modality_models(X_train_by_modality, y_train, model_factory=make_lgbm)

meta_features_val = build_meta_features(modality_models, X_val_by_modality)
meta_model = fit_meta_model(meta_features_val, y_val)

print("Learned meta-model weights (logistic-regression coefficients):")
pd.Series(meta_model.coef_[0], index=meta_features_val.columns).sort_values(ascending=False)

Learned meta-model weights (logistic-regression coefficients):


D:\Practise\Stock Market Analysis\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


macro_regime    1.132815
technical       0.111339
news            0.007097
fundamental    -0.163640
dtype: float64

## Step 5: Compare each individual modality vs the full ensemble, on the held-out test set

In [6]:
results = {}
for modality, model in modality_models.items():
    proba = model.predict_proba(X_test_by_modality[modality])[:, 1]
    pred = model.predict(X_test_by_modality[modality])
    results[modality] = classification_metrics(y_test.values, pred, proba)

ensemble_proba = ensemble_predict_proba(modality_models, meta_model, X_test_by_modality)
ensemble_pred = (ensemble_proba > 0.5).astype(int)
results["multimodal_ensemble"] = classification_metrics(y_test.values, ensemble_pred, ensemble_proba)

comparison = pd.DataFrame(results).T
comparison.sort_values("roc_auc", ascending=False)

,accuracy,precision,recall,f1,roc_auc,log_loss,brier_score
macro_regime,0.552632,0.492308,0.380952,0.429530,0.535377,0.726678,0.263559
multimodal_ensemble,0.557895,0.000000,0.000000,0.000000,0.531082,0.693837,0.250131
fundamental,0.510526,0.439189,0.386905,0.411392,0.515049,0.693257,0.250056
news,0.557895,0.000000,0.000000,0.000000,0.500000,0.691669,0.249261
technical,0.468421,0.424779,0.571429,0.487310,0.465970,0.788836,0.290486


## Step 6: Fair re-run of the Phase 5-style technical-only baseline on this exact split

(Phase 5's saved metrics used a wider date range, before the fundamentals/news/macro
availability constraints shrank the usable history -- re-running technical-only on the
*same* rows as the ensemble is the correct apples-to-apples DoD comparison.)

In [7]:
technical_only_model = modality_models["technical"]  # already trained on X_train_by_modality["technical"]
comparison.loc["technical_only_baseline"] = results["technical"]
print("Multimodal ensemble ROC-AUC:", round(results["multimodal_ensemble"]["roc_auc"], 4))
print("Technical-only baseline ROC-AUC:", round(results["technical"]["roc_auc"], 4))
print(
    "Ensemble outperforms technical-only baseline:",
    results["multimodal_ensemble"]["roc_auc"] > results["technical"]["roc_auc"],
)

Multimodal ensemble ROC-AUC: 0.5311
Technical-only baseline ROC-AUC: 0.466
Ensemble outperforms technical-only baseline: True


## Step 7: Walk-forward validation of H1-H5

For each expanding-window fold: fit each modality's base model (and a simple-average
ensemble across all four -- a lighter-weight combination than the learned meta-model
above, used here only to check whether combining modalities helps *consistently across
time*, not to re-derive optimal weights per fold) on data up to `train_end`, then
evaluate on the fold's test window.

In [8]:
folds = expanding_window_folds(dataset["date"], n_folds=4, min_train_frac=0.5)
print(f"{len(folds)} walk-forward folds:")
for train_end, test_start, test_end in folds:
    print(f"  train up to {pd.Timestamp(train_end).date()} | test {pd.Timestamp(test_start).date()} to {pd.Timestamp(test_end).date()}")

4 walk-forward folds:
  train up to 2025-07-28 | test 2025-07-29 to 2025-11-11
  train up to 2025-11-11 | test 2025-11-12 to 2026-02-24
  train up to 2026-02-24 | test 2026-02-25 to 2026-06-15
  train up to 2026-06-15 | test 2026-06-16 to 2026-09-24


In [9]:
fold_rows = []
for fold_idx, (train_end, test_start, test_end) in enumerate(folds):
    fold_train = dataset[dataset["date"] <= train_end]
    fold_test = dataset[(dataset["date"] >= test_start) & (dataset["date"] <= test_end)]
    if fold_test["next_day_direction"].nunique() < 2:
        continue  # roc_auc is undefined with a single class in the fold

    fold_X_train = {m: fold_train[cols] for m, cols in MULTIMODAL_FEATURE_GROUPS.items()}
    fold_X_test = {m: fold_test[cols] for m, cols in MULTIMODAL_FEATURE_GROUPS.items()}
    fold_models = fit_modality_models(fold_X_train, fold_train["next_day_direction"], model_factory=make_lgbm)

    fold_probas = {m: model.predict_proba(fold_X_test[m])[:, 1] for m, model in fold_models.items()}
    fold_probas["simple_average_ensemble"] = np.mean(list(fold_probas.values()), axis=0)

    for system, proba in fold_probas.items():
        pred = (proba > 0.5).astype(int)
        metrics = classification_metrics(fold_test["next_day_direction"].values, pred, proba)
        fold_rows.append({"fold": fold_idx, "system": system, **metrics})

walkforward_results = pd.DataFrame(fold_rows)
walkforward_summary = walkforward_results.groupby("system")["roc_auc"].agg(["mean", "std", "count"])
walkforward_summary.sort_values("mean", ascending=False)

,mean,std,count
system,,,
technical,0.537577,0.035409,4
simple_average_ensemble,0.509488,0.018707,4
news,0.500000,0.000000,4
macro_regime,0.499416,0.047088,4
fundamental,0.494549,0.022221,4


## Step 8: Per-regime accuracy breakdown (H4: does regime information improve stability?)

In [10]:
test_with_regime = test.copy()
test_with_regime["ensemble_pred"] = ensemble_pred
test_with_regime["regime_label"] = np.select(
    [
        (test_with_regime["is_bull"] == 1) & (test_with_regime["is_high_vol"] == 0),
        (test_with_regime["is_bull"] == 1) & (test_with_regime["is_high_vol"] == 1),
        (test_with_regime["is_bull"] == 0) & (test_with_regime["is_high_vol"] == 0),
        (test_with_regime["is_bull"] == 0) & (test_with_regime["is_high_vol"] == 1),
    ],
    ["bull_low_vol", "bull_high_vol", "bear_low_vol", "bear_high_vol"],
    default="unknown",
)
test_with_regime["correct"] = (test_with_regime["ensemble_pred"] == test_with_regime["next_day_direction"])
test_with_regime.groupby("regime_label")["correct"].agg(["mean", "count"])

,mean,count
regime_label,,
bear_high_vol,0.550000,60
bear_low_vol,0.523810,105
bull_high_vol,0.400000,5
bull_low_vol,0.580952,210


## Step 9: Save

In [11]:
comparison.to_csv(DATA_PROCESSED / "multimodal_test_metrics.csv")
walkforward_results.to_csv(DATA_PROCESSED / "multimodal_walkforward_results.csv", index=False)
print("Saved:", DATA_PROCESSED / "multimodal_test_metrics.csv")
print("Saved:", DATA_PROCESSED / "multimodal_walkforward_results.csv")

Saved: D:\Practise\Stock Market Analysis\data\processed\multimodal_test_metrics.csv
Saved: D:\Practise\Stock Market Analysis\data\processed\multimodal_walkforward_results.csv


## Reflection

**Actual results from this run** (single chronological test split / 4-fold walk-forward):

| system | test-set ROC-AUC | walk-forward mean ROC-AUC (std) |
|---|---|---|
| technical (baseline) | 0.466 | **0.538** (0.035) |
| fundamental | 0.515 | 0.495 (0.022) |
| news | 0.500 | 0.500 (0.000) |
| macro_regime | 0.535 | 0.499 (0.047) |
| multimodal ensemble | **0.531** | 0.509 (simple-average, 0.019) |

**H1 (technical > price-only):** already established in Phase 5; not re-litigated here.

**H2 (fundamentals add value):** ROC-AUC ~0.50-0.52 in both views -- roughly base-rate.
Fundamentals change at most once per fiscal year (after the reporting lag), so they act
as a slow-moving, near-constant-within-year signal against a *daily* target; this is
expected, not a bug (Phase 12's long-term-horizon model is the right place to test H2
properly).

**H3 (news adds value):** exactly 0.500 in both views, with zero standard deviation
across folds -- with only ~12 articles total, most rows have no news that day, so the
news modality is behaving almost exactly like the naive baseline. Purely a
data-availability limitation carried over from Phase 6, not a modeling failure.

**H4 (regime improves stability):** macro_regime scored best on the single test split
(0.535) but was close to base-rate on the walk-forward folds (0.499, and the joint
noisiest at std 0.047) -- i.e. it doesn't show a stable edge either; regime information
did not clearly reduce variance here. Worth revisiting once more history (and Phase 9's
proper backtest) is available.

**H5 (ensemble beats individual models):** **not confirmed** by this run. The multimodal
ensemble does beat the technical-only baseline on the single held-out test split (0.531
vs 0.466), which *technically* satisfies this phase's literal DoD comparison -- but that
single-split technical score (0.466) is unusually low compared to its own walk-forward
average (0.538), meaning the single split is likely an unrepresentative, noisy window
rather than evidence the ensemble is genuinely better. The walk-forward view, which
averages over 4 non-overlapping out-of-sample periods, is the more trustworthy signal
here, and it says the *opposite*: plain technical-only (0.538) outperforms the simple-average
ensemble (0.509) and every other individual modality. This is exactly the kind of result
the project plan warns about (Section 29: use walk-forward, not a single split) and is
reported honestly rather than cherry-picking the split that favors the more complex model.

**Why this happened, most likely:** fundamentals/news/macro_regime are currently weak or
near-uninformative modalities (see H2-H4 above), so stacking them alongside a genuinely
useful technical model mostly adds noise to the meta-model rather than signal. The fix is
not to force an ensemble narrative here, but to treat this as a real finding: multimodal
gains will likely need richer fundamentals/news coverage (Phase 6's data-source gap) before
they can consistently add value over technical-only.

**What's next:** Phase 8 (volatility model, probabilistic forecasting, risk engine) and
Phase 9 (proper backtesting with transaction costs) will show whether this holds up under
realistic trading assumptions. Any future ensemble work should prioritize closing the
news/fundamentals data gaps over further model tuning.